In [2]:
from pennylane import numpy as np
import pennylane as qml
import itertools
from itertools import combinations
from scipy.optimize import minimize
from math import comb
import qaoa_aux 
#importlib.reload(qaoa_aux)

In [3]:
def U_B(beta,edge_list):
    for edge_idx, (u, v) in enumerate(edge_list):
        theta = beta[edge_idx]
        qml.S(wires=u)
        qml.SingleExcitationMinus(-2 * theta,wires=[u, v])
        qml.adjoint(qml.S)(wires=u)

In [4]:
def U_C(gamma,cost_terms):
    # we have term index + cost terms so all the even numbered ZZ's
    for term_idx, (_, wires, coeff) in enumerate(cost_terms):
        theta = 2 * gamma[term_idx] * coeff
        # example for 4 consectuve z's
        #qml.PauliRot(theta, "ZZZZ", wires=[0,1,2,3])
        qml.PauliRot(theta,"Z" * len(wires),wires=list(wires))

In [5]:
def objective(gammas,betas,circuit,hyperedge_list,cost_terms,weights):
    expectation_values = circuit(gammas, betas)
    value = 0.0

    # weighted constant term for each hyperedge
    for edge_idx, edge in enumerate(hyperedge_list):
        k = len(edge)
        #w_e * 1 - (1/2^k-1)
        value += weights[edge_idx] * (1 - 1 / (2 ** (k - 1)))

    # coeff is weighted in build_cost_terms()
    for term_idx, (_, _, coeff) in enumerate(cost_terms):
        value += coeff * expectation_values[term_idx]

    return value

In [9]:
# Example small test hypergraph
hyperedge_list = [
    [0, 1, 2, 3]
]

n_wires = 4
num_layers = 1
seed = 42
example_weights = [1]

rng = np.random.default_rng(seed)

edge_list = qaoa_aux.clique_expansion(hyperedge_list)
cost_terms =  qaoa_aux.build_cost_terms(hyperedge_list,example_weights)

num_mixer_edges = len(edge_list)
num_cost_terms = len(cost_terms)

target_weight = n_wires // 2
initial_state = qaoa_aux.make_dicke_state(n_wires, target_weight)

dev = qml.device("default.qubit", wires=n_wires)

print("example of cost term quadruple Z:")
print(cost_terms[-1])

print("Mixer edges:", edge_list)
print("Number of mixer parameters/layer:", num_mixer_edges)
print("Number of cost parameters/layer:", num_cost_terms)

example of cost term quadruple Z:
(0, (0, 1, 2, 3), -0.125)
Mixer edges: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
Number of mixer parameters/layer: 6
Number of cost parameters/layer: 7


In [6]:

def run_qaoa(
    hyperedge_list,
    n,
    weights,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.01,
    max_trials=100
):
    n_wires=n
    assert len(weights)==len(hyperedge_list)
    # Build mixer edges and cost terms
    edge_list = qaoa_aux.clique_expansion(hyperedge_list)
    cost_terms = qaoa_aux.build_cost_terms(hyperedge_list,weights)

    # Balanced initial state
    target_weight = n_wires // 2
    initial_state = qaoa_aux.make_dicke_state(n_wires,target_weight)

    dev = qml.device("lightning.qubit",wires=n_wires)
    @qml.qnode(dev)
    def circuit(gammas, betas):
        qml.StatePrep(initial_state,wires=range(n_wires))

        for layer in range(num_layers):
            U_C(gammas[:, layer],cost_terms)
            U_B(betas[:, layer],edge_list)

        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables

    # Random parameter initialization
    rng = np.random.default_rng(seed)
    beta_params = np.array(scaling_constant * rng.random((len(edge_list), num_layers)),requires_grad=True)
    gamma_params = np.array(scaling_constant * rng.random((len(cost_terms), num_layers)),requires_grad=True)

    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_values = []
    initial_value = objective(gamma_params,beta_params,circuit,hyperedge_list,cost_terms,weights)
    objective_values.append(initial_value)

    print(f"Initial cut = {initial_value:.6f}")

    # Optimization
    for trial in range(max_trials):

        gamma_params, beta_params = opt.step(
            lambda g, b: objective(
                g,
                b,
                circuit,
                hyperedge_list,
                cost_terms,
                weights
            ),
            gamma_params,
            beta_params
        )

        current_value = objective(gamma_params,beta_params,circuit,hyperedge_list,cost_terms,weights)
        objective_values.append(current_value)

        if trial % 10 == 0:
            print(f"Iteration {trial:3d} | "f"cut = {current_value:.6f}")

    return {
        "gammas": gamma_params,
        "betas": beta_params,
        "objective_values": objective_values,
        "circuit": circuit,
        "edge_list": edge_list,
        "cost_terms": cost_terms,
        "n_wires": n_wires
    }

In [7]:
P = np.load("../data/poisson_16_12_3/P/03.npy")
w= np.load("../data/poisson_16_12_3/w/03.npy")

hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)

In [8]:

result = run_qaoa(
    hyperedge_list,
    weights=w,
    n=12,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.05,
    max_trials=300
)

Initial cut = 12.607519
Iteration   0 | cut = 12.310836
Iteration  10 | cut = 11.145354
Iteration  20 | cut = 10.976540
Iteration  30 | cut = 10.773577
Iteration  40 | cut = 10.688185
Iteration  50 | cut = 10.613247
Iteration  60 | cut = 10.569898
Iteration  70 | cut = 10.554420
Iteration  80 | cut = 10.542025
Iteration  90 | cut = 10.530749
Iteration 100 | cut = 10.515195
Iteration 110 | cut = 10.503369
Iteration 120 | cut = 10.493681
Iteration 130 | cut = 10.449277
Iteration 140 | cut = 10.081528
Iteration 150 | cut = 9.801338
Iteration 160 | cut = 9.680783
Iteration 170 | cut = 9.623745
Iteration 180 | cut = 9.571303
Iteration 190 | cut = 9.528026
Iteration 200 | cut = 9.517578
Iteration 210 | cut = 9.509980
Iteration 220 | cut = 9.503167
Iteration 230 | cut = 9.493566
Iteration 240 | cut = 9.485165
Iteration 250 | cut = 9.482565
Iteration 260 | cut = 9.482062
Iteration 270 | cut = 9.481682
Iteration 280 | cut = 9.481455
Iteration 290 | cut = 9.481421


In [9]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_3/P"
folder_w = "../data/poisson_16_12_3/w"


In [10]:
p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results = []

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [1, 2, 3]:

        print(f"\nRunning instance {instance} | p={p}")

        result = run_qaoa(hyperedge_list,weights=w,n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)

        final_objective = float(result["objective_values"][-1])

        initial_objective = float(result["objective_values"][0])

        results.append({
            "dataset": "poisson_16_12_3",
            "instance": instance,
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "initial_objective": initial_objective,
            "final_objective": final_objective,

            "gammas": np.array2string(np.asarray(result["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result["betas"]),separator=",")
        })



Running instance 01 | p=1
Initial cut = 13.113443
Iteration   0 | cut = 13.069956
Iteration  10 | cut = 12.403236
Iteration  20 | cut = 11.722198
Iteration  30 | cut = 11.108488
Iteration  40 | cut = 10.647644
Iteration  50 | cut = 10.307596
Iteration  60 | cut = 10.019577
Iteration  70 | cut = 9.769828
Iteration  80 | cut = 9.532827
Iteration  90 | cut = 9.292758
Iteration 100 | cut = 9.087447
Iteration 110 | cut = 8.934090
Iteration 120 | cut = 8.787560
Iteration 130 | cut = 8.645702
Iteration 140 | cut = 8.523062
Iteration 150 | cut = 8.396717
Iteration 160 | cut = 8.274021
Iteration 170 | cut = 8.160754
Iteration 180 | cut = 8.059417
Iteration 190 | cut = 7.980203
Iteration 200 | cut = 7.923616
Iteration 210 | cut = 7.881578
Iteration 220 | cut = 7.844198
Iteration 230 | cut = 7.813009
Iteration 240 | cut = 7.790265
Iteration 250 | cut = 7.769266
Iteration 260 | cut = 7.747057
Iteration 270 | cut = 7.724962
Iteration 280 | cut = 7.705080
Iteration 290 | cut = 7.688271

Running ins

In [11]:
df = pd.DataFrame(results)
csv_path = "poisson_16_12_3_qaoa_p1_p2_p3.csv"

df.to_csv(csv_path,index=False)

print(f"saved results to: {csv_path}")

saved results to: poisson_16_12_3_qaoa_p1_p2_p3.csv
